# Heap / Priority Queue in Python

A heap is a complete binary tree where the parent is always <= children (min-heap). Python's `heapq` implements a min-heap.

## Topics Covered
- Heap as Priority Queue
- heapq module (heappush, heappop, heapify, nlargest, nsmallest)
- Min-Heap vs Max-Heap (negation trick)
- Top-K Elements Pattern
- Heap Sort
- K-Way Merge
- Median in Stream

---
## 1. heapq Module Basics

Python's `heapq` provides min-heap operations. All operations are O(log n) for push/pop.

In [ ]:
import heapq

# Creating a heap
heap = []
heapq.heappush(heap, 10)
heapq.heappush(heap, 1)
heapq.heappush(heap, 5)
heapq.heappush(heap, 3)
print(f"Heap: {heap}")  # [1, 3, 5, 10] (not fully sorted, but min at top)

# Pop min
min_val = heapq.heappop(heap)
print(f"Pop min: {min_val}")  # 1
print(f"Heap after pop: {heap}")

# Peek at min (without popping)
print(f"Peek min: {heap[0]}")

# heapify - convert list to heap in O(n)
arr = [30, 10, 50, 20, 40]
heapq.heapify(arr)
print(f"After heapify: {arr}")  # [10, 20, 50, 30, 40]

# push + pop combined
result = heapq.heappushpop(arr, 5)  # Push 5, then pop and return min
print(f"heappushpop(arr, 5): {result}, heap: {arr}")

# Pop and push new
result = heapq.heapreplace(arr, 25)  # Pop min, then push 25
print(f"heapreplace(arr, 25): {result}, heap: {arr}")

---
## 2. nlargest / nsmallest

Efficiently get the K largest/smallest elements from an iterable.

In [ ]:
data = [10, 30, 50, 20, 40, 5, 25]

# Top 3 largest
top3 = heapq.nlargest(3, data)
print(f"Top 3 largest: {top3}")  # [50, 40, 30]

# Bottom 3 smallest
bottom3 = heapq.nsmallest(3, data)
print(f"Bottom 3 smallest: {bottom3}")  # [5, 10, 20]

# Works with custom keys
people = [
    {"name": "Alice", "age": 30},
    {"name": "Bob", "age": 25},
    {"name": "Charlie", "age": 35},
]
youngest = heapq.nsmallest(2, people, key=lambda x: x["age"])
print(f"Youngest 2: {youngest}")

---
## 3. Max-Heap Using Negation

Python only provides min-heap. For max-heap, negate values on push/pop.

In [ ]:
class MaxHeap:
    def __init__(self):
        self.heap = []

    def push(self, val):
        heapq.heappush(self.heap, -val)

    def pop(self):
        return -heapq.heappop(self.heap)

    def peek(self):
        return -self.heap[0] if self.heap else None

    def __len__(self):
        return len(self.heap)

maxh = MaxHeap()
maxh.push(10); maxh.push(30); maxh.push(20)
print(f"Max peek: {maxh.peek()}")   # 30
print(f"Max pop: {maxh.pop()}")     # 30
print(f"Max pop: {maxh.pop()}")     # 20

# Alternative: heap of tuples (-priority, value)
h = []
heapq.heappush(h, (-30, 30))
heapq.heappush(h, (-10, 10))
heapq.heappush(h, (-20, 20))
priority, value = heapq.heappop(h)
print(f"Pop max via tuple: value={value}, priority={-priority}")

---
## 4. Top-K Elements Pattern

When you need the K largest/smallest elements, use a heap of size K.

In [ ]:
def find_kth_largest(nums, k):
    """Return kth largest element using min-heap of size k."""
    heap = nums[:k]
    heapq.heapify(heap)
    for num in nums[k:]:
        if num > heap[0]:
            heapq.heapreplace(heap, num)
    return heap[0]

def top_k_frequent(nums, k):
    """Return k most frequent elements."""
    from collections import Counter
    freq = Counter(nums)
    # Min-heap of size k
    heap = []
    for num, count in freq.items():
        heapq.heappush(heap, (count, num))
        if len(heap) > k:
            heapq.heappop(heap)
    return [num for _, num in heap]

print(f"3rd largest in [3,2,1,5,6,4]: {find_kth_largest([3,2,1,5,6,4], 3)}")
print(f"Top 2 frequent: {top_k_frequent([1,1,1,2,2,3], 2)}")
# Time: O(n log k), Space: O(k)

---
## 5. Heap Sort

Build heap, repeatedly extract min.

In [ ]:
def heap_sort(arr):
    """Sort array using heap."""
    heapq.heapify(arr)
    sorted_arr = []
    while arr:
        sorted_arr.append(heapq.heappop(arr))
    return sorted_arr

arr = [3, 1, 4, 1, 5, 9, 2, 6]
print(f"Heap sort: {heap_sort(arr)}")
# Time: O(n log n), Space: O(n)

---
## 6. K-Way Merge

Merge K sorted arrays/lists using a heap.

In [ ]:
def merge_k_sorted(lists):
    """Merge K sorted lists into one sorted list."""
    heap = []
    # Push first element of each list
    for i, lst in enumerate(lists):
        if lst:
            heapq.heappush(heap, (lst[0], i, 0))
    result = []
    while heap:
        val, list_idx, elem_idx = heapq.heappop(heap)
        result.append(val)
        # Push next element from same list
        if elem_idx + 1 < len(lists[list_idx]):
            heapq.heappush(heap, (
                lists[list_idx][elem_idx + 1],
                list_idx,
                elem_idx + 1
            ))
    return result

lists = [
    [1, 4, 7],
    [2, 5, 8],
    [3, 6, 9]
]
print(f"K-way merge: {merge_k_sorted(lists)}")
# Time: O(n log k), Space: O(k) for heap

---
## 7. Find Median from Data Stream

Use two heaps: max-heap for left half, min-heap for right half. Maintain balanced sizes.

In [ ]:
class MedianFinder:
    """Find median from a stream of numbers."""
    def __init__(self):
        self.small = []  # Max-heap (negated) for left half
        self.large = []  # Min-heap for right half

    def addNum(self, num):
        heapq.heappush(self.small, -num)  # Add to max-heap
        # Ensure every element in small <= every element in large
        if self.small and self.large and (-self.small[0]) > self.large[0]:
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)
        # Balance sizes: small can have at most 1 more than large
        if len(self.small) > len(self.large) + 1:
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)
        elif len(self.large) > len(self.small):
            val = heapq.heappop(self.large)
            heapq.heappush(self.small, -val)

    def findMedian(self):
        if len(self.small) > len(self.large):
            return -self.small[0]
        return (-self.small[0] + self.large[0]) / 2

mf = MedianFinder()
for num in [1, 2, 3, 4, 5]:
    mf.addNum(num)
    print(f"After adding {num}, median: {mf.findMedian()}")
# Time: O(log n) per add, O(1) for findMedian

---
## LeetCode Problems

### LeetCode 215 - Kth Largest Element in an Array

**Problem:** Find the kth largest element in an unsorted array.

In [ ]:
def findKthLargest(nums, k):
    heap = nums[:k]
    heapq.heapify(heap)
    for num in nums[k:]:
        if num > heap[0]:
            heapq.heapreplace(heap, num)
    return heap[0]

# Test
print(findKthLargest([3,2,1,5,6,4], 2))  # 5
print(findKthLargest([3,2,3,1,2,4,5,5,6], 4))  # 4
# Time: O(n log k), Space: O(k)

### LeetCode 347 - Top K Frequent Elements

**Problem:** Return the k most frequent elements.

In [ ]:
from collections import Counter

def topKFrequent(nums, k):
    freq = Counter(nums)
    heap = []
    for num, count in freq.items():
        heapq.heappush(heap, (count, num))
        if len(heap) > k:
            heapq.heappop(heap)
    return [num for _, num in heap]

# Test
print(topKFrequent([1,1,1,2,2,3], 2))  # [1, 2]
print(topKFrequent([1], 1))            # [1]
# Time: O(n log k), Space: O(n)

### LeetCode 23 - Merge k Sorted Lists

**Problem:** Merge K sorted linked lists into one sorted list.

In [ ]:
# Definition for singly-linked list
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def mergeKLists(lists):
    heap = []
    for i, lst in enumerate(lists):
        if lst:
            heapq.heappush(heap, (lst.val, i, lst))
    dummy = curr = ListNode()
    while heap:
        val, i, node = heapq.heappop(heap)
        curr.next = ListNode(val)
        curr = curr.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next
# Time: O(n log k), Space: O(k)

### LeetCode 295 - Find Median from Data Stream

**Problem:** Find median from a stream of integers.

In [ ]:
class MedianFinder:
    def __init__(self):
        self.small = []
        self.large = []

    def addNum(self, num):
        heapq.heappush(self.small, -num)
        if self.small and self.large and (-self.small[0]) > self.large[0]:
            heapq.heappush(self.large, -heapq.heappop(self.small))
        if len(self.small) > len(self.large) + 1:
            heapq.heappush(self.large, -heapq.heappop(self.small))
        elif len(self.large) > len(self.small):
            heapq.heappush(self.small, -heapq.heappop(self.large))

    def findMedian(self):
        if len(self.small) > len(self.large):
            return -self.small[0]
        return (-self.small[0] + self.large[0]) / 2.0

# Time: O(log n) per addNum, O(1) per findMedian
# Space: O(n)

---
## Summary

| Pattern | Data Structure | Time | Key Insight |
|---|---|---|---|
| Kth Largest | Min-heap of size k | O(n log k) | Heap of k largest elements |
| Top K Frequent | Min-heap of size k | O(n log k) | Heap with (freq, element) |
| K-Way Merge | Heap of K elements | O(n log k) | Push next from popped list |
| Median in Stream | Two heaps | O(log n) | Balance left (max) and right (min) |
| Heap Sort | Complete heap | O(n log n) | Repeated extract-min |